# Week 4 – Day 1: GNN Environment and Graph Data Preparation

## Objective

Prepare the validated Week 3 spatial graph for Graph Neural Network modeling using PyTorch Geometric.

The finalized graph contains property-level node features, spatial relationships, spatial embeddings, and property-price targets.

The goal of this day is to create a clean and validated PyTorch Geometric graph dataset for the GNN training stage.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import torch
import torch_geometric

from torch_geometric.data import Data

from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
print("Python environment ready")
print("PyTorch version:", torch.__version__)
print("PyTorch Geometric version:", torch_geometric.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

Python environment ready
PyTorch version: 2.13.0+cpu
PyTorch Geometric version: 2.8.0.post1
CUDA available: False
Using CPU


In [3]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data" / "processed" / "week-3"

NODE_FILE = DATA_DIR / "final_graph_nodes.csv"
EDGE_FILE = DATA_DIR / "final_graph_edges.csv"
TARGET_FILE = DATA_DIR / "final_graph_targets.csv"

OUTPUT_DIR = DATA_DIR / "gnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Data directory: E:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed\week-3
Output directory: E:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed\week-3\gnn


In [4]:
required_files = [NODE_FILE,EDGE_FILE,TARGET_FILE]

for file_path in required_files:
    print(file_path.name, "->", file_path.exists())

final_graph_nodes.csv -> True
final_graph_edges.csv -> True
final_graph_targets.csv -> True


In [5]:
nodes_df = pd.read_csv(NODE_FILE)
edges_df = pd.read_csv(EDGE_FILE)
targets_df = pd.read_csv(TARGET_FILE)

print("Node dataset:", nodes_df.shape)
print("Edge dataset:", edges_df.shape)
print("Target dataset:", targets_df.shape)

Node dataset: (21613, 26)
Edge dataset: (108065, 3)
Target dataset: (21613, 2)


In [6]:
print("Node columns:")
print(nodes_df.columns.tolist())

Node columns:
['node_id', 'id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'House Age', 'yr_built', 'yr_renovated', 'lat', 'long', 'lat_scaled', 'long_scaled', 'mean_neighbor_distance_scaled', 'median_neighbor_distance_scaled', 'min_neighbor_distance_scaled', 'max_neighbor_distance_scaled', 'std_neighbor_distance_scaled', 'neighbor_count']


In [7]:
print("Target columns:")
print(targets_df.columns.tolist())

TARGET_COLUMN = "price"

assert TARGET_COLUMN in targets_df.columns
assert TARGET_COLUMN not in nodes_df.columns

print("✅ Target price is stored separately.")

Target columns:
['node_id', 'price']
✅ Target price is stored separately.


In [8]:
nodes_df = nodes_df.sort_values("node_id").reset_index(drop=True)

targets_df = targets_df.sort_values("node_id").reset_index(drop=True)

print(nodes_df["node_id"].head().tolist())
print(targets_df["node_id"].head().tolist())

[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4]


In [9]:
assert nodes_df["node_id"].is_unique
assert targets_df["node_id"].is_unique

assert set(nodes_df["node_id"]) == set(targets_df["node_id"])

print("✅ Node and target IDs are aligned.")

✅ Node and target IDs are aligned.


In [10]:
node_id_to_index = {
    node_id: index
    for index, node_id in enumerate(nodes_df["node_id"])
}

print("Mapped nodes:", len(node_id_to_index))

Mapped nodes: 21613


In [11]:
edge_source = edges_df["source"].map(node_id_to_index)

edge_target = edges_df["target"].map(node_id_to_index)

assert edge_source.notna().all()
assert edge_target.notna().all()

edge_index = torch.tensor(np.vstack([edge_source.to_numpy(),edge_target.to_numpy()]),dtype=torch.long)

print("edge_index shape:", edge_index.shape)

edge_index shape: torch.Size([2, 108065])


In [12]:
assert edge_index.min() >= 0
assert edge_index.max() < len(nodes_df)

print("Minimum index:", edge_index.min().item())
print("Maximum index:", edge_index.max().item())

print("✅ Edge indices validated.")

Minimum index: 0
Maximum index: 21612
✅ Edge indices validated.


In [13]:
feature_columns = [
    col
    for col in nodes_df.columns
    if col not in ["node_id", "id"]
]

print("Number of node features:", len(feature_columns))

print("\nFeatures:")
for feature in feature_columns:
    print("-", feature)

Number of node features: 24

Features:
- bedrooms
- bathrooms
- sqft_living
- sqft_lot
- floors
- waterfront
- view
- condition
- grade
- sqft_above
- sqft_basement
- House Age
- yr_built
- yr_renovated
- lat
- long
- lat_scaled
- long_scaled
- mean_neighbor_distance_scaled
- median_neighbor_distance_scaled
- min_neighbor_distance_scaled
- max_neighbor_distance_scaled
- std_neighbor_distance_scaled
- neighbor_count


In [14]:
X = nodes_df[feature_columns].copy()

y = targets_df[TARGET_COLUMN].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (21613, 24)
y shape: (21613,)


In [15]:
print("Missing feature values:",X.isnull().sum().sum())

print("Missing target values:",y.isnull().sum())

assert X.isnull().sum().sum() == 0
assert y.isnull().sum() == 0

print("✅ No missing values detected.")

Missing feature values: 0
Missing target values: 0
✅ No missing values detected.


In [16]:
rng = np.random.default_rng(42)

num_nodes = len(nodes_df)

indices = np.arange(num_nodes)

rng.shuffle(indices)

train_end = int(0.70 * num_nodes)
val_end = int(0.85 * num_nodes)

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

print("Training nodes:", len(train_indices))
print("Validation nodes:", len(val_indices))
print("Test nodes:", len(test_indices))

Training nodes: 15129
Validation nodes: 3242
Test nodes: 3242


In [17]:
train_mask = torch.zeros(num_nodes,dtype=torch.bool)

val_mask = torch.zeros(num_nodes,dtype=torch.bool)

test_mask = torch.zeros(num_nodes,dtype=torch.bool)

train_mask[train_indices] = True
val_mask[val_indices] = True
test_mask[test_indices] = True

assert train_mask.sum() == len(train_indices)
assert val_mask.sum() == len(val_indices)
assert test_mask.sum() == len(test_indices)

assert not (train_mask & val_mask).any()

assert not (train_mask & test_mask).any()

assert not (val_mask & test_mask).any()

print("✅ Train/validation/test masks validated.")

✅ Train/validation/test masks validated.


In [18]:
scaler = StandardScaler()

X_train = X.iloc[train_indices]

scaler.fit(X_train)

X_scaled = scaler.transform(X)

print("Scaled feature matrix:", X_scaled.shape)

Scaled feature matrix: (21613, 24)


In [19]:
x_tensor = torch.tensor(X_scaled,dtype=torch.float32)

y_tensor = torch.tensor(y.to_numpy(),dtype=torch.float32).view(-1, 1)

print("Node tensor:", x_tensor.shape)
print("Target tensor:", y_tensor.shape)

Node tensor: torch.Size([21613, 24])
Target tensor: torch.Size([21613, 1])


In [20]:
graph_data = Data(x=x_tensor,edge_index=edge_index,y=y_tensor,train_mask=train_mask,val_mask=val_mask,test_mask=test_mask)

print(graph_data)

Data(x=[21613, 24], edge_index=[2, 108065], y=[21613, 1], train_mask=[21613], val_mask=[21613], test_mask=[21613])


In [21]:
assert graph_data.num_nodes == 21613
assert graph_data.num_node_features == 24

assert graph_data.edge_index.shape == (2,108065)

assert graph_data.y.shape == (21613,1)

assert graph_data.train_mask.sum() == len(train_indices)

assert graph_data.val_mask.sum() == len(val_indices)

assert graph_data.test_mask.sum() == len(test_indices)

print("✅ PyTorch Geometric graph validation passed.")

✅ PyTorch Geometric graph validation passed.


In [22]:
assert graph_data.edge_index.min().item() >= 0

assert (graph_data.edge_index.max().item() < graph_data.num_nodes)

assert not (graph_data.edge_index[0] == graph_data.edge_index[1]).any()

print("✅ Edge range validated.")
print("✅ No self-loops detected.")

✅ Edge range validated.
✅ No self-loops detected.


In [23]:
print("=" * 60)
print("WEEK 4 DAY 1 – GNN DATA PREPARATION")
print("=" * 60)

print(f"Nodes: {graph_data.num_nodes:,}")
print(f"Edges: {graph_data.edge_index.shape[1]:,}")
print(f"Node features: {graph_data.num_node_features}")
print(f"Training nodes: {train_mask.sum().item():,}")
print(f"Validation nodes: {val_mask.sum().item():,}")
print(f"Test nodes: {test_mask.sum().item():,}")
print("Target: price")

print("\nValidation:")
print("✓ Final Week 3 graph loaded")
print("✓ Node features validated")
print("✓ Target separated")
print("✓ Edge indices converted")
print("✓ Train/validation/test masks created")
print("✓ Training-only feature scaling applied")
print("✓ PyTorch Geometric graph created")
print("✓ Edge indices validated")
print("✓ Self-loops checked")

print("\n🎉 WEEK 4 DAY 1 COMPLETED")

WEEK 4 DAY 1 – GNN DATA PREPARATION
Nodes: 21,613
Edges: 108,065
Node features: 24
Training nodes: 15,129
Validation nodes: 3,242
Test nodes: 3,242
Target: price

Validation:
✓ Final Week 3 graph loaded
✓ Node features validated
✓ Target separated
✓ Edge indices converted
✓ Train/validation/test masks created
✓ Training-only feature scaling applied
✓ PyTorch Geometric graph created
✓ Edge indices validated
✓ Self-loops checked

🎉 WEEK 4 DAY 1 COMPLETED


In [24]:
GRAPH_OUTPUT = (OUTPUT_DIR / "housing_graph_data.pt")

torch.save(graph_data,GRAPH_OUTPUT)

print("✅ GNN graph saved successfully.")
print(GRAPH_OUTPUT.resolve())

✅ GNN graph saved successfully.
E:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed\week-3\gnn\housing_graph_data.pt


In [25]:
loaded_graph = torch.load(GRAPH_OUTPUT,weights_only=False)

print(loaded_graph)

Data(x=[21613, 24], edge_index=[2, 108065], y=[21613, 1], train_mask=[21613], val_mask=[21613], test_mask=[21613])


In [26]:
assert loaded_graph.num_nodes == 21613
assert loaded_graph.num_node_features == 24
assert loaded_graph.edge_index.shape[1] == 108065

print("✅ Saved GNN graph successfully reloaded.")

✅ Saved GNN graph successfully reloaded.
